# SuperMimic on Colab

One-time setup:
- Runtime, change runtime type, pick a GPU.
- Colab Secrets (key icon on the left): add `WANDB_API_KEY` and allow notebook access.

Run the cells top to bottom. Motions and data live on Drive in `supermimic-official`. The Python environment lives in `/content` and is reinstalled every session.

## 1. GPU check

In [ ]:
import subprocess
out = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,driver_version,memory.total", "--format=csv,noheader"],
    capture_output=True, text=True,
).stdout.strip()
print(out)
assert out, "No GPU found. Change the runtime type to a GPU."
driver = out.splitlines()[0].split(",")[1].strip()
assert int(driver.split(".")[0]) >= 580, (
    f"NVIDIA driver {driver} is too old. The pinned torch build (CUDA 13) needs driver 580 or newer."
)

## 2. Drive and repos

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

STORE = "/content/drive/MyDrive/supermimic-official"
WS = "/content/supermimic_ws"

# Sibling clones: pyproject.toml resolves mjlab from ../mjlab_research.
!mkdir -p {WS} {STORE}/data
!if [ -d {WS}/mjlab_research ]; then git -C {WS}/mjlab_research pull --ff-only; else git clone --branch main https://github.com/xSurus/mjlab_research.git {WS}/mjlab_research; fi
!if [ -d {WS}/supermimic ]; then git -C {WS}/supermimic pull --ff-only; else git clone --branch thesis https://github.com/xSurus/supermimic.git {WS}/supermimic; fi
# data/ is not in .gitignore, so it stays outside the repo and cells use absolute paths.

## 3. Environment

In [ ]:
!pip install -q uv
!cd {WS}/supermimic && env -u PYTHONPATH uv sync --locked --group dev
!cd {WS}/supermimic && env -u PYTHONPATH uv run python -c "import torch; print(torch.__version__); assert torch.cuda.is_available()"

## 4. Weights & Biases

In [ ]:
import os
from google.colab import userdata

key = userdata.get("WANDB_API_KEY")
assert key, "Add your W&B API key to Colab Secrets as WANDB_API_KEY"
os.environ["WANDB_API_KEY"] = key

## 5. Tests

In [ ]:
!cd {WS}/supermimic && MUJOCO_GL=disable env -u PYTHONPATH uv run pytest -q -rs

## 6. Motions

Downloads the LAFAN1 G1 retargeted CSVs and converts them to npz on Drive. Existing files are skipped.

In [ ]:
!cd {WS}/supermimic && env -u PYTHONPATH uv run --with huggingface_hub hf download lvhaidong/LAFAN1_Retargeting_Dataset --repo-type dataset --include "g1/*" --local-dir {STORE}/data/raw/lafan1
!cd {WS}/supermimic && env -u PYTHONPATH uv run python colab/build_motions.py --csv-dir {STORE}/data/raw/lafan1/g1 --out-dir {STORE}/data/npz/lafan1

## 7. Smoke run

In [ ]:
# The first update compiles with torch.compile. On Colab's 2 CPU cores this can sit
# silent for 20+ minutes. That is not a hang, wait for it.
# For a quick check only, add TORCHDYNAMO_DISABLE=1 before "env" to skip compilation.
# Updates are then slower.
!cd {WS}/supermimic && env -u PYTHONPATH uv run src/supermimic/scripts/train.py SuperMimic-GenCoord-G1 --motion-file {STORE}/data/npz/lafan1/walk1_subject1.npz --num-envs 256 --runner.max-iterations 300 --runner.run-name colab-smoke

## 8. Full run

In [ ]:
# !cd {WS}/supermimic && env -u PYTHONPATH uv run src/supermimic/scripts/train.py SuperMimic-GenCoord-G1 --motion-file '{STORE}/data/npz/lafan1/*.npz' --num-envs 4096

## 9. Play

In [ ]:
# Replace the placeholders with the W&B entity and run id.
# !cd {WS}/supermimic && env -u PYTHONPATH uv run src/supermimic/scripts/play.py SuperMimic-GenCoord-G1 --wandb-path <entity>/supermimic/<run id>